# Trail Surface Type and Obstacle Identification

The following code uses **CLIP** to identify trail surface type and **GroundingDINO** to identify obstacles. It then uses **SAM** to draw contours around the obstacles.

## Step 1: Setup

### Install dependencies for GroundingDINO, SAM, and Transformers

In [ ]:
!pip install -q pillow supervision transformers groundingdino-py
!pip install -q "git+https://github.com/facebookresearch/segment-anything.git"

In [ ]:
import os
import time
import torch
import cv2
import numpy as np
import transformers
import supervision as sv
from PIL import Image
from IPython.display import display
from google.colab import files
from groundingdino.util.inference import load_model, load_image, predict
from segment_anything import sam_model_registry, SamPredictor
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

### Transformers patches

Newer versions of `transformers` are missing/changed methods that GroundingDINO depends on. Patch them so it works.

In [ ]:
# fix missing get_head_mask in newer transformers
if not hasattr(transformers.models.bert.modeling_bert.BertModel, "get_head_mask"):
    transformers.models.bert.modeling_bert.BertModel.get_head_mask = \
        lambda self, head_mask, num_hidden_layers, is_attention_chunked=False: [None] * num_hidden_layers

# fix argument mismatch for get_extended_attention_mask
original_get_mask = transformers.modeling_utils.ModuleUtilsMixin.get_extended_attention_mask

def patched_get_extended_attention_mask(self, attention_mask, input_shape, *args, **kwargs):
    return original_get_mask(self, attention_mask, input_shape)

transformers.models.bert.modeling_bert.BertPreTrainedModel.get_extended_attention_mask = patched_get_extended_attention_mask

### Load GroundingDINO

In [ ]:
os.makedirs("weights", exist_ok=True)

# download weights and config if they don't exist
if not os.path.exists("weights/groundingdino_swint_ogc.pth"):
    print("Downloading GroundingDINO weights...")
    !wget -q -O weights/groundingdino_swint_ogc.pth https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
if not os.path.exists("weights/GroundingDINO_SwinT_OGC.py"):
    print("Downloading GroundingDINO config...")
    !wget -q -O weights/GroundingDINO_SwinT_OGC.py https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py

if 'dino_model' not in locals():
    print("Loading GroundingDINO Model...")
    dino_model = load_model(
        "weights/GroundingDINO_SwinT_OGC.py",
        "weights/groundingdino_swint_ogc.pth",
        device=device
    )
    print("GroundingDINO Model loaded.")

### Load CLIP

In [ ]:
if 'clip_model' not in locals():
    print("Loading CLIP Model...")
    clip_model_name = "openai/clip-vit-base-patch32"
    clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
    clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
    print("CLIP Model loaded.")

### Load SAM

In [ ]:
if not os.path.exists("weights/sam_vit_h_4b8939.pth"):
    print("Downloading Segment Anything (SAM) weights...")
    !wget -q -O weights/sam_vit_h_4b8939.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

if 'sam_predictor' not in locals():
    print(f"Loading SAM on {device}...")
    sam_checkpoint = "weights/sam_vit_h_4b8939.pth"
    sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)
    sam.to(device=device)
    sam_predictor = SamPredictor(sam)
    print("SAM loaded.")

## Step 2: Define surface ID labels

*Kept separate so they can be easily modified and tested.*

In [ ]:
# define labels for surface identification
surface_categories = {
    "Gravel": [
        "a photo of a gravel trail",
        "a photo of a trail made of loose, small crushed rocks",
        "a photo of a crushed stone path"
    ],
    "Paved": [
        "a photo of a paved asphalt path",
        "a photo of a smooth concrete trail",
        "an artificial sidewalk with clean, straight edges",
        "a smooth surface designed for bicycle tires and car wheels"
    ],
    "Boardwalk": [
        "a photo of a wide wooden boardwalk",
        "a photo of a flat, accessible path built with multiple wooden planks",
        "a photo of a wooden bridge wide enough for a wheelchair"
    ],
    "Rocky": [
        "a photo of a rocky hiking trail",
        "a photo of uneven ground covered in large boulders and stones"
    ]
}

# Flatten dictionary into single list of phrases for CLIP
all_phrases = []
phrase_to_category = {}
for category, phrases in surface_categories.items():
    for phrase in phrases:
        all_phrases.append(phrase)
        phrase_to_category[phrase] = category

## Step 3: Upload images

* Can select multiple photos at a time

In [ ]:
print("\nSelect trail photos to process...")
uploaded_images = files.upload()

if uploaded_images:
    # GroundingDINO Parameters
    TEXT_PROMPT = " log . rock . trail . "
    BOX_THRESHOLD  = 0.30
    TEXT_THRESHOLD = 0.25

    print(f"\nProcessing {len(uploaded_images)} images...")

## Step 4: Main processing loop

* Bottom-center crop
* Surface type ID
* Obstacle ID

In [ ]:
for image_filename in uploaded_images.keys():
    start_time = time.time()
    print(f"\n{'-'*50}\nAnalyzing: {image_filename}\n{'-'*50}")

    # ---------------------------------------------------------
    # 1: Predict surface type (CLIP w/ crop)
    # ---------------------------------------------------------
    original_img = Image.open(image_filename).convert("RGB")
    width, height = original_img.size

    # crop params: Left 20%, Top 60%, Right 80%, Bottom 100%
    left = int(width * 0.2)
    top = int(height * 0.6)
    right = int(width * 0.8)
    bottom = height

    trail_crop = original_img.crop((left, top, right, bottom))

    # Show trail segment that CLIP is evaluating
    print("Cropped segment that CLIP is evaluating for surface type:")
    display(trail_crop.resize((300, int(300 * (trail_crop.height / trail_crop.width)))))

    # Run CLIP inference on the cropped image
    clip_inputs = clip_processor(text=all_phrases, images=trail_crop, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)

    probs_np = clip_outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]

    category_scores = {cat: 0.0 for cat in surface_categories.keys()}
    for phrase, score in zip(all_phrases, probs_np):
        cat = phrase_to_category[phrase]
        if score > category_scores[cat]:
            category_scores[cat] = score

    predicted_surface = max(category_scores, key=category_scores.get)
    surface_confidence = category_scores[predicted_surface] * 100

    # ---------------------------------------------------------
    # 2: Obstacle detection/ID (GroundingDINO)
    # ---------------------------------------------------------
    image_source, image_tensor = load_image(image_filename)
    h, w, _ = image_source.shape

    # TODO: Could add more obstacle types
    HAZARD_PROMPT = " log . rock . boulder . puddle . "
    boxes, logits, phrases = predict(
        model=dino_model,
        image=image_tensor,
        caption=HAZARD_PROMPT,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        device=device
    )

    # ---------------------------------------------------------
    # 3: Segment hazards & annotate image (SAM + Supervision)
    # ---------------------------------------------------------
    if len(boxes) > 0:
        # Keep math on CPU to avoid device mismatch errors
        boxes_unnormalized = boxes * torch.Tensor([w, h, w, h])

        # Manual conversion to xyxy coordinates (avoids supervision library bugs)
        boxes_xyxy = torch.zeros_like(boxes_unnormalized)
        boxes_xyxy[:, 0] = boxes_unnormalized[:, 0] - boxes_unnormalized[:, 2] / 2
        boxes_xyxy[:, 1] = boxes_unnormalized[:, 1] - boxes_unnormalized[:, 3] / 2
        boxes_xyxy[:, 2] = boxes_unnormalized[:, 0] + boxes_unnormalized[:, 2] / 2
        boxes_xyxy[:, 3] = boxes_unnormalized[:, 1] + boxes_unnormalized[:, 3] / 2

        # Move boxes to GPU specifically for SAM
        sam_predictor.set_image(image_source)
        transformed_boxes = sam_predictor.transform.apply_boxes_torch(boxes_xyxy, image_source.shape[:2]).to(device)

        masks, _, _ = sam_predictor.predict_torch(
            point_coords=None,
            point_labels=None,
            boxes=transformed_boxes,
            multimask_output=False,
        )

        # Set up Supervision tools for drawing
        detections = sv.Detections(
            xyxy=boxes_xyxy.cpu().numpy(),
            mask=masks.squeeze(1).cpu().numpy(),
            class_id=np.arange(len(phrases))
        )

        mask_annotator = sv.MaskAnnotator()
        box_annotator = sv.BoxAnnotator()

        # Apply colored overlays and bounding boxes
        annotated_image = mask_annotator.annotate(scene=image_source.copy(), detections=detections)
        annotated_image = box_annotator.annotate(scene=annotated_image, detections=detections)

        for xyxy, phrase in zip(detections.xyxy, phrases):
            cv2.putText(annotated_image, phrase, (int(xyxy[0]), int(xyxy[1])-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        final_display_img = Image.fromarray(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
    else:
        final_display_img = Image.fromarray(cv2.cvtColor(image_source, cv2.COLOR_BGR2RGB))

    # ---------------------------------------------------------
    # 4: Display final output
    # ---------------------------------------------------------
    final_display_img.thumbnail((800, 800))
    print("Full Hazard Analysis:")
    display(final_display_img)

    print(f"Predicted Surface: **{predicted_surface.upper()}** ({surface_confidence:.2f}% confidence)")

    if len(boxes) > 0:
        print(f"Hazards Outlined: {', '.join(phrases)}")
    else:
        print("Hazards Outlined: None detected.")

    print("\nSurface Category Leaderboard:")
    sorted_cats = sorted(category_scores.items(), key=lambda item: item[1], reverse=True)
    for cat, score in sorted_cats[:4]:
        print(f"- {cat}: {score*100:.1f}%")

    print(f"\n(Processed in {time.time() - start_time:.2f} seconds)")

---
# Test code below

## Surface type ID only

*Using CLIP to ID trail surface type, **not** cropping image.*

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from google.colab import files

# loading model & processor
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

# running on gpu if possible
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"CLIP loaded on {device}")

In [ ]:
# TODO: could play around w these labels to find an optimal combination.

# Define labels - dict where key is the final CSV label (gravel, paved, etc.)
# and the value is a list of prompts corresponding to each key
surface_categories = {
    "Gravel": [
        "a photo of a gravel trail",
        "a photo of a trail made of loose, small crushed rocks",
        "a photo of a crushed stone path"
    ],
    "Paved": [
        "a photo of a paved asphalt path",
        "a photo of a smooth concrete trail",
        "an artificial sidewalk with clean, straight edges",
        "a smooth surface designed for bicycle tires and car wheels"
    ],
    "Bog Bridge": [
        "a photo of a narrow bog bridge",
        "a photo of a single wooden log placed on the trail to walk on",
        "a photo of a narrow wooden plank over mud, difficult for a wheelchair"
    ],
    "Boardwalk": [
        "a photo of a wide wooden boardwalk",
        "a photo of a flat, accessible path built with multiple wooden planks",
        "a photo of a wooden bridge wide enough for a wheelchair"
    ],
    "Woodchips": [
        "a photo of woodchips on a trail",
        "a photo of a path covered in shredded bark and mulch"
    ],
    "Rocky": [
        "a photo of a rocky hiking trail",
        "a photo of uneven ground covered in large boulders and stones"
    ]
}

# flatten dictionary into single list of phrases for CLIP
all_phrases = []
phrase_to_category = {}

for category, phrases in surface_categories.items():
    for phrase in phrases:
        all_phrases.append(phrase)
        phrase_to_category[phrase] = category

In [ ]:
# Upload photos
print("Select your trail photos...")
uploaded = files.upload()

for filename in uploaded.keys():
    image = Image.open(filename).convert("RGB")  # load & process images

    # run CLIP inference on all phrases
    inputs = processor(text=all_phrases, images=image, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    # get confidence scores and apply Softmax
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1)
    probs_np = probs.cpu().numpy()[0]

    # aggregate the scores by category
    category_scores = {cat: 0.0 for cat in surface_categories.keys()}

    for phrase, score in zip(all_phrases, probs_np):
        cat = phrase_to_category[phrase]
        # if this phrase scored higher than the current max for this category, update it
        if score > category_scores[cat]:
            category_scores[cat] = score

    # find the winning category
    winning_category = max(category_scores, key=category_scores.get)
    confidence = category_scores[winning_category] * 100

    # output results
    print(f"\n========================================")
    print(f"--- Result for {filename} ---")
    display(image.resize((300, 200)))  # previews image
    print(f"Predicted Surface: **{winning_category.upper()}**")
    print(f"Confidence Score: {confidence:.2f}%\n")

    # print the aggregated scores for all categories
    print("Category Leaderboard:")
    # sort categories by score descending
    sorted_cats = sorted(category_scores.items(), key=lambda item: item[1], reverse=True)
    for cat, score in sorted_cats:
        print(f"- {cat}: {score*100:.1f}%")